In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

In [2]:
CSV_PATH = r"C:\Users\jashw\OneDrive\Documents\2-1\DAV\DAV Project\cleaned data\indian_roads_dataset_cleaned.csv"
df = pd.read_csv(CSV_PATH)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Column names:", df.columns.tolist())
display(df.head())

Rows: 20000
Columns: 24
Column names: ['accident_id', 'city', 'state', 'latitude', 'longitude', 'date', 'time', 'hour', 'day_of_week', 'is_weekend', 'road_type', 'lanes', 'traffic_signal', 'weather', 'visibility', 'temperature', 'traffic_density', 'cause', 'accident_severity', 'vehicles_involved', 'casualties', 'is_peak_hour', 'festival', 'risk_score']


,accident_id,city,state,latitude,longitude,date,time,hour,day_of_week,is_weekend,road_type,lanes,traffic_signal,weather,visibility,temperature,traffic_density,cause,accident_severity,vehicles_involved,casualties,is_peak_hour,festival,risk_score
0,0,Pune,Maharashtra,18.680827,73.930388,22-10-2023,05:00,5,Sunday,1,highway,3,1,fog,low,32,high,weather,fatal,2,2,0,No Festival,0.85
1,1,Mumbai,Maharashtra,18.817732,72.790846,21-05-2023,04:00,4,Sunday,1,urban,4,0,clear,high,34,low,weather,major,4,3,0,No Festival,0.10
2,2,Mumbai,Maharashtra,19.096889,72.819424,10-07-2024,13:00,13,Wednesday,0,urban,3,0,fog,low,21,medium,weather,minor,1,1,0,No Festival,0.45
3,3,Chandigarh,Punjab,30.787805,76.847507,30-03-2025,11:00,11,Sunday,1,urban,1,1,fog,low,30,high,distraction,minor,5,2,0,No Festival,0.65
4,4,Chennai,Tamil Nadu,12.965155,80.283313,25-01-2024,16:00,16,Thursday,0,highway,3,1,clear,high,24,low,distraction,minor,2,1,0,No Festival,0.10


## 3. Data types and descriptive statistics

In [3]:
df.info()
display(df.describe(include="all").T)

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   accident_id        20000 non-null  int64  
 1   city               20000 non-null  str    
 2   state              20000 non-null  str    
 3   latitude           20000 non-null  float64
 4   longitude          20000 non-null  float64
 5   date               20000 non-null  str    
 6   time               20000 non-null  str    
 7   hour               20000 non-null  int64  
 8   day_of_week        20000 non-null  str    
 9   is_weekend         20000 non-null  int64  
 10  road_type          20000 non-null  str    
 11  lanes              20000 non-null  int64  
 12  traffic_signal     20000 non-null  int64  
 13  weather            20000 non-null  str    
 14  visibility         20000 non-null  str    
 15  temperature        20000 non-null  int64  
 16  traffic_density    20000 non-null

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
accident_id,20000.0,NaN,NaN,NaN,9999.5,5773.647028,0.0,4999.75,9999.5,14999.25,19999.0
city,20000,8,Chandigarh,2577,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,20000,7,Maharashtra,5009,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,20000.0,NaN,NaN,NaN,20.389207,6.165791,12.800172,13.198653,18.812008,28.402467,30.79996
longitude,20000.0,NaN,NaN,NaN,78.17333,4.485967,72.700017,73.997979,77.297,80.111089,88.499861
date,20000,1201,03-11-2022,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
time,20000,24,02:00,888,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hour,20000.0,NaN,NaN,NaN,11.4872,6.945563,0.0,5.0,12.0,18.0,23.0
day_of_week,20000,7,Monday,2966,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_weekend,20000.0,NaN,NaN,NaN,0.28615,0.451972,0.0,0.0,0.0,1.0,1.0


## 4. Data quality: missing values and duplicates

In [4]:
quality = pd.DataFrame({"missing_count": df.isna().sum(), "missing_percent": (df.isna().mean()*100).round(2), "unique_values": df.nunique(dropna=True)})
display(quality.sort_values(["missing_count", "unique_values"], ascending=[False, True]))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Duplicate accident_id values:", int(df["accident_id"].duplicated().sum()))

,missing_count,missing_percent,unique_values
is_weekend,0,0.0,2
traffic_signal,0,0.0,2
is_peak_hour,0,0.0,2
road_type,0,0.0,3
weather,0,0.0,3
visibility,0,0.0,3
traffic_density,0,0.0,3
accident_severity,0,0.0,3
cause,0,0.0,5
vehicles_involved,0,0.0,5


Duplicate rows: 0
Duplicate accident_id values: 0


## 5. Prepare date fields

In [5]:
df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y", errors="coerce")
df["year_month"] = df["date"].dt.to_period("M").astype("string")
print("Unparsed dates:", int(df["date"].isna().sum()))
print("Date range:", df["date"].min(), "to", df["date"].max())

Unparsed dates: 0
Date range: 2022-01-01 00:00:00 to 2025-04-15 00:00:00


## 6. Accident records by state

In [6]:
by_state = df.groupby("state").size().rename("accident_records").to_frame()
by_state["share_percent"] = (by_state["accident_records"] / len(df) * 100).round(2)
display(by_state.sort_values("accident_records", ascending=False))

,accident_records,share_percent
state,,
Maharashtra,5009,25.04
Punjab,2577,12.88
Tamil Nadu,2575,12.88
West Bengal,2559,12.80
Karnataka,2438,12.19
Delhi,2433,12.16
Telangana,2409,12.04


## 7. Accident records by city: top 20

In [7]:
by_city = df["city"].value_counts().head(20).rename_axis("city").to_frame("accident_records")
by_city["share_percent"] = (by_city["accident_records"] / len(df) * 100).round(2)
display(by_city)

,accident_records,share_percent
city,,
Chandigarh,2577,12.88
Chennai,2575,12.88
Kolkata,2559,12.80
Pune,2517,12.58
Mumbai,2492,12.46
Bangalore,2438,12.19
Delhi,2433,12.16
Hyderabad,2409,12.04


## 8. Accident severity: counts and percentages

In [8]:
severity = df["accident_severity"].value_counts(dropna=False).rename("accident_records").to_frame()
severity["percent"] = (severity["accident_records"] / len(df) * 100).round(2)
display(severity)

,accident_records,percent
accident_severity,,
minor,11025,55.12
major,5988,29.94
fatal,2987,14.94


## 9. Reported causes: counts and percentages

In [9]:
causes = df["cause"].value_counts(dropna=False).rename("accident_records").to_frame()
causes["percent"] = (causes["accident_records"] / len(df) * 100).round(2)
display(causes)

,accident_records,percent
cause,,
distraction,4026,20.13
overspeeding,4025,20.12
weather,3997,19.98
drunk driving,3978,19.89
poor road,3974,19.87


## 10. Accidents by hour

In [10]:
by_hour = df.groupby("hour").size().reindex(range(24), fill_value=0).rename("accident_records").to_frame()
by_hour["percent"] = (by_hour["accident_records"] / len(df) * 100).round(2)
display(by_hour)
print("Hour with the most records:", int(by_hour["accident_records"].idxmax()))

,accident_records,percent
hour,,
0,840,4.20
1,859,4.30
2,888,4.44
3,824,4.12
4,805,4.03
5,827,4.14
6,856,4.28
7,786,3.93
8,849,4.24


Hour with the most records: 2


## 11. Accidents by day of week

In [11]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_weekday = df["day_of_week"].value_counts().reindex(weekday_order, fill_value=0).rename("accident_records").to_frame()
by_weekday["percent"] = (by_weekday["accident_records"] / len(df) * 100).round(2)
display(by_weekday)

,accident_records,percent
day_of_week,,
Monday,2966,14.83
Tuesday,2871,14.36
Wednesday,2726,13.63
Thursday,2835,14.18
Friday,2879,14.40
Saturday,2867,14.34
Sunday,2856,14.28


## 12. Weekend and peak-hour comparisons

In [12]:
for col, labels in [("is_weekend", {0:"Weekday", 1:"Weekend"}), ("is_peak_hour", {0:"Not peak hour", 1:"Peak hour"})]:
    summary = df[col].map(labels).value_counts().rename("accident_records").to_frame()
    summary["percent"] = (summary["accident_records"] / len(df) * 100).round(2)
    print("\n", col)
    display(summary)


 is_weekend


,accident_records,percent
is_weekend,,
Weekday,14277,71.39
Weekend,5723,28.62



 is_peak_hour


,accident_records,percent
is_peak_hour,,
Not peak hour,15052,75.26
Peak hour,4948,24.74


## 13. Weather, visibility, and traffic density

In [13]:
for col in ["weather", "visibility", "traffic_density"]:
    summary = df[col].value_counts(dropna=False).rename("accident_records").to_frame()
    summary["percent"] = (summary["accident_records"] / len(df) * 100).round(2)
    print("\n", col)
    display(summary)


 weather


,accident_records,percent
weather,,
clear,6690,33.45
rain,6677,33.38
fog,6633,33.16



 visibility


,accident_records,percent
visibility,,
low,9987,49.94
high,6690,33.45
medium,3323,16.61



 traffic_density


,accident_records,percent
traffic_density,,
low,7067,35.34
high,7034,35.17
medium,5899,29.49


## 14. Casualties and vehicles involved

In [14]:
display(df[["casualties", "vehicles_involved"]].describe().T)
casualties_by_severity = df.groupby("accident_severity").agg(accident_records=("accident_id", "size"), mean_casualties=("casualties", "mean"), median_casualties=("casualties", "median"), mean_vehicles=("vehicles_involved", "mean")).round(2)
display(casualties_by_severity.sort_values("mean_casualties", ascending=False))

,count,mean,std,min,25%,50%,75%,max
casualties,20000.0,1.72645,1.489104,0.0,1.0,1.0,3.0,5.0
vehicles_involved,20000.0,2.99200,1.415534,1.0,2.0,3.0,4.0,5.0


,accident_records,mean_casualties,median_casualties,mean_vehicles
accident_severity,,,,
fatal,2987,3.00,3.0,3.00
minor,11025,1.51,1.0,3.00
major,5988,1.49,1.0,2.97


## 15. Compare average casualties by state

In [15]:
state_casualties = df.groupby("state").agg(accident_records=("accident_id", "size"), total_casualties=("casualties", "sum"), average_casualties=("casualties", "mean"), median_casualties=("casualties", "median")).round(2)
display(state_casualties.sort_values("average_casualties", ascending=False))

,accident_records,total_casualties,average_casualties,median_casualties
state,,,,
Tamil Nadu,2575,4567,1.77,1.0
West Bengal,2559,4508,1.76,1.0
Maharashtra,5009,8716,1.74,1.0
Delhi,2433,4175,1.72,1.0
Punjab,2577,4377,1.70,1.0
Karnataka,2438,4128,1.69,1.0
Telangana,2409,4058,1.68,1.0


## 16. Risk score summaries

In [16]:
print("Overall risk score summary:")
display(df["risk_score"].describe().to_frame("risk_score"))
print("\nMean risk score by severity:")
display(df.groupby("accident_severity")["risk_score"].agg(["count", "mean", "median", "min", "max"]).round(3).sort_values("mean", ascending=False))
print("\nMean risk score by weather:")
display(df.groupby("weather")["risk_score"].agg(["count", "mean", "median"]).round(3).sort_values("mean", ascending=False))

Overall risk score summary:


,risk_score
count,20000.000000
mean,0.437585
std,0.218130
min,0.100000
25%,0.250000
50%,0.450000
75%,0.600000
max,1.000000



Mean risk score by severity:


,count,mean,median,min,max
accident_severity,,,,,
fatal,2987,0.608,0.65,0.3,1.0
minor,11025,0.408,0.45,0.1,0.9
major,5988,0.407,0.45,0.1,0.9



Mean risk score by weather:


,count,mean,median
weather,,,
fog,6633,0.589,0.60
rain,6677,0.488,0.45
clear,6690,0.237,0.25


## 17. Monthly accident record totals

In [17]:
monthly = df.dropna(subset=["date"]).groupby("year_month").size().rename("accident_records").to_frame()
monthly["change_from_previous_month"] = monthly["accident_records"].diff()
monthly["percent_change"] = (monthly["accident_records"].pct_change() * 100).round(2)
display(monthly)

,accident_records,change_from_previous_month,percent_change
year_month,,,
2022-01,499,NaN,NaN
2022-02,474,-25.0,-5.01
2022-03,520,46.0,9.70
2022-04,527,7.0,1.35
2022-05,548,21.0,3.98
2022-06,510,-38.0,-6.93
2022-07,548,38.0,7.45
2022-08,483,-65.0,-11.86
2022-09,509,26.0,5.38


## 18. Numerical correlations

In [18]:
numeric_cols = ["latitude", "longitude", "hour", "lanes", "traffic_signal", "temperature", "vehicles_involved", "casualties", "is_weekend", "is_peak_hour", "risk_score"]
display(df[numeric_cols].corr(numeric_only=True).round(3))

,latitude,longitude,hour,lanes,traffic_signal,temperature,vehicles_involved,casualties,is_weekend,is_peak_hour,risk_score
latitude,1.000,0.009,-0.001,-0.008,0.000,-0.002,-0.004,-0.005,-0.002,-0.002,0.001
longitude,0.009,1.000,-0.002,0.003,-0.004,-0.007,0.004,0.008,-0.002,0.009,-0.001
hour,-0.001,-0.002,1.000,0.001,0.009,-0.006,-0.004,-0.004,-0.004,0.172,0.122
lanes,-0.008,0.003,0.001,1.000,-0.001,-0.007,0.002,0.001,0.010,0.007,-0.002
traffic_signal,0.000,-0.004,0.009,-0.001,1.000,0.004,0.003,0.014,0.004,0.008,0.002
temperature,-0.002,-0.007,-0.006,-0.007,0.004,1.000,-0.004,-0.006,-0.011,-0.013,-0.002
vehicles_involved,-0.004,0.004,-0.004,0.002,0.003,-0.004,1.000,0.551,0.007,0.002,0.001
casualties,-0.005,0.008,-0.004,0.001,0.014,-0.006,0.551,1.000,0.001,0.004,0.120
is_weekend,-0.002,-0.002,-0.004,0.010,0.004,-0.011,0.007,0.001,1.000,0.002,0.004
is_peak_hour,-0.002,0.009,0.172,0.007,0.008,-0.013,0.002,0.004,0.002,1.000,0.437


## 19. Export summary tables (optional)

In [19]:
by_state.sort_values("accident_records", ascending=False).to_csv("accidents_by_state_summary.csv")
severity.to_csv("accidents_by_severity_summary.csv")
causes.to_csv("accidents_by_cause_summary.csv")
print("Summary CSV files saved in the notebook working directory:", __import__("os").getcwd())

Summary CSV files saved in the notebook working directory: c:\Users\jashw\OneDrive\Documents\2-1\DAV\DAV Project\data analysis


## Interpretation note

The counts describe records in this dataset. Group differences and correlations are descriptive; by themselves they do not show that a factor caused an accident. The supplied `risk_score` is summarized as recorded in the file.